# Postprocess Experiment 1 (Backdoor CATE)


In [2]:
import os
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)









In [3]:
import csv
import os
import hashlib
import numpy as np
from scipy.stats import binomtest


def load_summary(path):
    rows = []
    with open(path, newline='', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    return rows


def as_float(x):
    try:
        return float(x)
    except Exception:
        return float('nan')


def parse_n(row):
    try:
        return int(float(row.get('n')))
    except Exception:
        return None


def dgp_sort_key(s):
    try:
        parts = s.split('_')
        return int(parts[-1])
    except Exception:
        return s


def dgp_label(s):
    try:
        parts = s.split('_')
        idx = parts[-1]
        return r"\mathcal{D}_{%s}" % idx
    except Exception:
        return s


def method_label(m):
    if m is None:
        return ""
    label = str(m)
    if label.lower() == "aipw":
        return "DR"
    return label



def orthogonal_cell(m):
    label = method_label(m).lower() if m is not None else ''
    if label.startswith('dr') or label in ('aipw', 'dr'):
        return r"\cmark"
    return r"\xmark"
def mean_std(values):
    arr = np.array([v for v in values if not np.isnan(v)], dtype=float)
    if arr.size == 0:
        return np.nan, np.nan
    mean = float(arr.mean())
    std = float(arr.std(ddof=1)) if arr.size > 1 else 0.0
    return mean, std


def build_metric_stats(rows, methods, dgps, value_key):
    means = []
    stds = []
    for m in methods:
        row_means = []
        row_stds = []
        for d in dgps:
            vals = [as_float(r.get(value_key, '')) for r in rows if r.get('method') == m and r.get('dgp') == d]
            mean, std = mean_std(vals)
            row_means.append(mean)
            row_stds.append(std)
        means.append(row_means)
        stds.append(row_stds)
    return means, stds


def build_status_table(rows, methods, dgps):
    table = []
    for m in methods:
        row = []
        for d in dgps:
            subset = [r for r in rows if r.get('method') == m and r.get('dgp') == d]
            total = len(subset)
            ok = sum(1 for r in subset if r.get('status') == 'ok')
            cell = f"{ok}/{total}" if total > 0 else '0/0'
            row.append(cell)
        table.append(row)
    return table


def load_npz_arrays(artifacts_dir, cache):
    if not artifacts_dir:
        return None
    if artifacts_dir in cache:
        return cache[artifacts_dir]
    path = os.path.join(artifacts_dir, 'arrays.npz')
    if not os.path.exists(path):
        cache[artifacts_dir] = None
        return None
    with np.load(path, allow_pickle=False) as data:
        arrays = {k: data[k] for k in data.files}
    cache[artifacts_dir] = arrays
    return arrays



def select_eval_indices(n, dgp, rep, k, X=None, clip_pct=0.05):
    idx_all = np.arange(n, dtype=int)
    idx_pool = idx_all
    if X is not None and clip_pct is not None and clip_pct > 0:
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        X = X[:n]
        lo = np.quantile(X, clip_pct, axis=0)
        hi = np.quantile(X, 1.0 - clip_pct, axis=0)
        mask = np.ones(X.shape[0], dtype=bool)
        for j in range(X.shape[1]):
            mask &= (X[:, j] >= lo[j]) & (X[:, j] <= hi[j])
        idx_pool = np.where(mask)[0]
    if idx_pool.size == 0:
        idx_pool = idx_all
    if k is None or k <= 0 or k >= idx_pool.size:
        return np.sort(idx_pool)
    key = f"{dgp}|{rep}|{k}|{clip_pct}"
    seed = int(hashlib.sha1(key.encode("utf-8")).hexdigest()[:8], 16)
    rng = np.random.default_rng(seed)
    idx = rng.choice(idx_pool, size=k, replace=False)
    return np.sort(idx)


def compute_ci_arrays(mean, var, z=1.96):
    sd = np.sqrt(np.maximum(var, 0))
    lo = mean - z * sd
    hi = mean + z * sd
    return lo, hi, hi - lo


def compute_cate_ci_run_stats(arrays, mode, dgp, rep, k_eval=None, clip_pct=0.05, z=1.96):
    if arrays is None:
        return np.nan, 0, 0
    mean_key = f"cate_mean_{mode}"
    var_key = f"cate_var_{mode}"
    if mean_key not in arrays or var_key not in arrays or 'cate_true' not in arrays:
        return np.nan, 0, 0
    mean = np.asarray(arrays[mean_key], dtype=float)
    var = np.asarray(arrays[var_key], dtype=float)
    true = np.asarray(arrays['cate_true'], dtype=float)
    n = min(mean.size, var.size, true.size)
    if n == 0:
        return np.nan, 0, 0
    mean = mean[:n]
    var = var[:n]
    true = true[:n]
    X = arrays.get("cate_X_eval", None)
    if X is not None:
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        X = X[:n]
    if k_eval is not None and k_eval > 0:
        idx = select_eval_indices(n, dgp, rep, k_eval, X=X, clip_pct=clip_pct)
        mean = mean[idx]
        var = var[idx]
        true = true[idx]
    lo, hi, length = compute_ci_arrays(mean, var, z=z)
    mask = (~np.isnan(length)) & (~np.isnan(true))
    if mask.any():
        mean_len = float(np.mean(length[mask]))
    else:
        mean_len = np.nan
    covered = int(np.sum((lo <= true) & (true <= hi) & mask))
    total = int(np.sum(mask))
    return mean_len, covered, total


def build_cate_ci_stats(rows, methods, dgps, mode, cache, k_eval=None, clip_pct=0.05):
    means = []
    stds = []
    for m in methods:
        row_means = []
        row_stds = []
        for d in dgps:
            vals = []
            for r in rows:
                if r.get('method') != m or r.get('dgp') != d:
                    continue
                if r.get('status') != 'ok':
                    continue
                arrays = load_npz_arrays(r.get('artifacts_dir', ''), cache)
                rep_val = None
                try:
                    rep_val = int(float(r.get("rep")))
                except Exception:
                    rep_val = r.get("rep")
                mean_len, _, total = compute_cate_ci_run_stats(arrays, mode, d, rep_val, k_eval, clip_pct)
                if total > 0 and not np.isnan(mean_len):
                    vals.append(mean_len)
            mean, std = mean_std(vals)
            row_means.append(mean)
            row_stds.append(std)
        means.append(row_means)
        stds.append(row_stds)
    return means, stds


def build_cate_coverage_stats(rows, methods, dgps, mode, cache, k_eval=None, clip_pct=0.05):
    cov = []
    cov_lower = []
    cov_upper = []
    counts = []
    for m in methods:
        row_cov = []
        row_lower = []
        row_upper = []
        row_counts = []
        for d in dgps:
            total = 0
            covered = 0
            for r in rows:
                if r.get('method') != m or r.get('dgp') != d:
                    continue
                if r.get('status') != 'ok':
                    continue
                arrays = load_npz_arrays(r.get('artifacts_dir', ''), cache)
                rep_val = None
                try:
                    rep_val = int(float(r.get("rep")))
                except Exception:
                    rep_val = r.get("rep")
                _, run_covered, run_total = compute_cate_ci_run_stats(arrays, mode, d, rep_val, k_eval, clip_pct)
                covered += run_covered
                total += run_total
            if total > 0:
                p = covered / total
                ci = binomtest(covered, total).proportion_ci(confidence_level=0.95, method='exact')
                lower = float(ci.low)
                upper = float(ci.high)
            else:
                p = float('nan')
                lower = float('nan')
                upper = float('nan')
            row_cov.append(p)
            row_lower.append(lower)
            row_upper.append(upper)
            row_counts.append((covered, total))
        cov.append(row_cov)
        cov_lower.append(row_lower)
        cov_upper.append(row_upper)
        counts.append(row_counts)
    return cov, cov_lower, cov_upper, counts


def apply_mask_to_stats(means, stds, mask):
    masked_means = []
    masked_stds = []
    for i in range(len(means)):
        row_means = []
        row_stds = []
        for j in range(len(means[i])):
            row_means.append(means[i][j])
            row_stds.append(stds[i][j])
        masked_means.append(row_means)
        masked_stds.append(row_stds)
    return masked_means, masked_stds, mask


def format_coverage_cell(mean, lo, hi, is_best, fmt='{:.3f}', ci_fmt='{:.3f}'):
    if np.isnan(mean) or np.isnan(lo) or np.isnan(hi):
        return '$NA$'
    text = f"{fmt.format(mean)}"
    ci_text = f"\\scriptsize ({ci_fmt.format(lo)}-{ci_fmt.format(hi)})"
    if is_best:
        text = f"\\textbf{{{text}}}"
    return f"${text}$ {ci_text}"


def table_to_latex_coverage(methods, dgps, means, lowers, uppers, target=0.95, fmt='{:.3f}', ci_fmt='{:.3f}'):
    cols = 'l' + 'c' * (len(dgps) + 1)
    lines = []
    lines.append(r'\begin{tabular}{' + cols + '}')
    lines.append(r'\hline')
    header = 'Strategy & Orthogonal & ' + ' & '.join([f"${dgp_label(d)}$" for d in dgps]) + ' \\\\ '
    lines.append(header)
    lines.append(r'\hline')
    for i, m in enumerate(methods):
        row_cells = []
        for j, d in enumerate(dgps):
            col_vals = [means[k][j] for k in range(len(methods))]
            finite_vals = [v for v in col_vals if not np.isnan(v)]
            best = False
            if finite_vals:
                best_val = min(finite_vals, key=lambda v: abs(v - target))
                best = np.isclose(means[i][j], best_val, rtol=1e-9, atol=1e-12)
            row_cells.append(format_coverage_cell(means[i][j], lowers[i][j], uppers[i][j], best, fmt=fmt, ci_fmt=ci_fmt))
        lines.append(method_label(m).upper() + ' & ' + orthogonal_cell(m) + ' & ' + ' & '.join(row_cells) + ' \\\\ ')
    lines.append(r'\hline')
    lines.append(r'\end{tabular}')
    return '\n'.join(lines)


def format_cell(mean, std, is_best, fmt='{:.4f}', masked=False):
    if np.isnan(mean):
        return '$NA$'
    text = f"{fmt.format(mean)} ({fmt.format(std)})"
    if is_best:
        text = f"\\textbf{{{text}}}"
    if masked:
        text = f"\\textcolor{{gray}}{{\\cancel{{{text}}}}}"
    return f"${text}$"


def table_to_latex_metric(methods, dgps, means, stds, fmt='{:.4f}', mask=None):
    cols = 'l' + 'c' * (len(dgps) + 1)
    lines = []
    lines.append(r'\begin{tabular}{' + cols + '}')
    lines.append(r'\hline')
    header = 'Strategy & Orthogonal & ' + ' & '.join([f"${dgp_label(d)}$" for d in dgps]) + ' \\\\ '
    lines.append(header)
    lines.append(r'\hline')
    for i, m in enumerate(methods):
        row_cells = []
        for j, d in enumerate(dgps):
            col_means = []
            for k in range(len(methods)):
                if mask is not None and not mask[k][j]:
                    continue
                col_means.append(means[k][j])
            finite_means = [v for v in col_means if not np.isnan(v)]
            best = False
            if finite_means:
                min_mean = min(finite_means)
                if mask is None or mask[i][j]:
                    best = np.isclose(means[i][j], min_mean, rtol=1e-9, atol=1e-12)
            masked = mask is not None and not mask[i][j]
            row_cells.append(format_cell(means[i][j], stds[i][j], best, fmt=fmt, masked=masked))
        lines.append(method_label(m).upper() + ' & ' + orthogonal_cell(m) + ' & ' + ' & '.join(row_cells) + ' \\\\ ')
    lines.append(r'\hline')
    lines.append(r'\end{tabular}')
    return '\n'.join(lines)

def table_to_latex_ci(methods, dgps, means, stds, fmt='{:.3f}', mask=None):
    return table_to_latex_metric(methods, dgps, means, stds, fmt=fmt, mask=mask)


def table_to_latex_text(methods, dgps, table):
    cols = 'l' + 'c' * (len(dgps) + 1)
    lines = []
    lines.append(r'\begin{tabular}{' + cols + '}')
    lines.append(r'\hline')
    header = 'Strategy & Orthogonal & ' + ' & '.join([f"${dgp_label(d)}$" for d in dgps]) + ' \\\\ '
    lines.append(header)
    lines.append(r'\hline')
    for m, row in zip(methods, table):
        row_cells = [f"${cell}$" for cell in row]
        lines.append(method_label(m).upper() + ' & ' + orthogonal_cell(m) + ' & ' + ' & '.join(row_cells) + ' \\\\ ')
    lines.append(r'\hline')
    lines.append(r'\end{tabular}')
    return '\n'.join(lines)

def save_text(path, text):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(text)
        f.write('\n')










In [4]:

# experiment_id = 'exp1_backdoor_cate_attempt_1'
# experiment_id = 'exp1_backdoor_cate_attempt_1_TEST'
experiment_id = 'exp1_backdoor_cate_attempt_1'

results_root = 'results'
outputs_root = 'outputs'
summary_path = os.path.join(results_root, experiment_id, 'summary.csv')

rows = load_summary(summary_path)
methods = sorted({r.get('method') for r in rows})
dgps = sorted({r.get('dgp') for r in rows}, key=dgp_sort_key)

dgps = [d for d in dgps if d != 'dgp_0']

n_list = sorted({parse_n(r) for r in rows if parse_n(r) is not None})

print('Methods:', methods)
print('DGPS:', dgps)
print('Sample sizes:', n_list)










Methods: ['aipw', 'ipw', 'ra']
DGPS: ['dgp_1', 'dgp_2', 'dgp_3', 'dgp_4', 'dgp_5', 'dgp_6', 'dgp_7', 'dgp_8', 'dgp_9']
Sample sizes: [1000]


In [5]:
K_EVAL = 100  # number of CATE eval points per run (before clipping)
CLIP_PCT = 0.05  # clip 5%% lower/upper tails per covariate

tables_dir_base = os.path.join(outputs_root, experiment_id, 'tables')


def tables_dir_for_n(n):
    if len(n_list) > 1:
        return os.path.join(tables_dir_base, f'n_{n}')
    return tables_dir_base


for n in n_list:
    rows_n = [r for r in rows if parse_n(r) == n]
    print(f'=== n={n} ({len(rows_n)} rows) ===')

    vi_means, vi_stds = build_metric_stats(rows_n, methods, dgps, 'cate_mse_vi')
    an_means, an_stds = build_metric_stats(rows_n, methods, dgps, 'cate_mse_analytic')
    status_ok = build_status_table(rows_n, methods, dgps)

    cache = {}
    ci_vi_means, ci_vi_stds = build_cate_ci_stats(rows_n, methods, dgps, 'vi', cache, k_eval=K_EVAL, clip_pct=CLIP_PCT)
    ci_an_means, ci_an_stds = build_cate_ci_stats(rows_n, methods, dgps, 'analytic', cache, k_eval=K_EVAL, clip_pct=CLIP_PCT)
    cov_vi, cov_vi_lower, cov_vi_upper, cov_vi_counts = build_cate_coverage_stats(rows_n, methods, dgps, 'vi', cache, k_eval=K_EVAL, clip_pct=CLIP_PCT)
    cov_an, cov_an_lower, cov_an_upper, cov_an_counts = build_cate_coverage_stats(rows_n, methods, dgps, 'analytic', cache, k_eval=K_EVAL, clip_pct=CLIP_PCT)

    tables_dir = tables_dir_for_n(n)

    latex_vi = table_to_latex_metric(methods, dgps, vi_means, vi_stds, fmt='{:.4f}')
    latex_an = table_to_latex_metric(methods, dgps, an_means, an_stds, fmt='{:.4f}')
    latex_status = table_to_latex_text(methods, dgps, status_ok)

    save_text(os.path.join(tables_dir, 'cate_mse_vi.txt'), latex_vi)
    save_text(os.path.join(tables_dir, 'cate_mse_analytic.txt'), latex_an)
    save_text(os.path.join(tables_dir, 'status_ok.txt'), latex_status)

    latex_ci_vi = table_to_latex_ci(methods, dgps, ci_vi_means, ci_vi_stds, fmt='{:.3f}')
    latex_ci_an = table_to_latex_ci(methods, dgps, ci_an_means, ci_an_stds, fmt='{:.3f}')
    save_text(os.path.join(tables_dir, 'ci_length_vi.txt'), latex_ci_vi)
    save_text(os.path.join(tables_dir, 'ci_length_analytic.txt'), latex_ci_an)

    latex_cov_vi = table_to_latex_coverage(methods, dgps, cov_vi, cov_vi_lower, cov_vi_upper, target=0.95, fmt='{:.3f}', ci_fmt='{:.3f}')
    latex_cov_an = table_to_latex_coverage(methods, dgps, cov_an, cov_an_lower, cov_an_upper, target=0.95, fmt='{:.3f}', ci_fmt='{:.3f}')
    save_text(os.path.join(tables_dir, 'ci_coverage_vi.txt'), latex_cov_vi)
    save_text(os.path.join(tables_dir, 'ci_coverage_analytic.txt'), latex_cov_an)

    mask_vi = [[(cov_vi_upper[i][j] >= 0.95) if not np.isnan(cov_vi_upper[i][j]) else False for j in range(len(dgps))] for i in range(len(methods))]
    mask_an = [[(cov_an_upper[i][j] >= 0.95) if not np.isnan(cov_an_upper[i][j]) else False for j in range(len(dgps))] for i in range(len(methods))]

    vi_masked_means, vi_masked_stds, vi_mask = apply_mask_to_stats(ci_vi_means, ci_vi_stds, mask_vi)
    an_masked_means, an_masked_stds, an_mask = apply_mask_to_stats(ci_an_means, ci_an_stds, mask_an)

    latex_ci_length_masked_vi = table_to_latex_ci(methods, dgps, vi_masked_means, vi_masked_stds, fmt='{:.3f}', mask=vi_mask)
    latex_ci_length_masked_an = table_to_latex_ci(methods, dgps, an_masked_means, an_masked_stds, fmt='{:.3f}', mask=an_mask)
    save_text(os.path.join(tables_dir, 'ci_length_masked_vi.txt'), latex_ci_length_masked_vi)
    save_text(os.path.join(tables_dir, 'ci_length_masked_analytic.txt'), latex_ci_length_masked_an)

    paper_mode = 'vi'  # set to 'vi' or 'analytic'
    if paper_mode == 'vi':
        paper_means, paper_stds = ci_vi_means, ci_vi_stds
    else:
        paper_means, paper_stds = ci_an_means, ci_an_stds
    latex_paper = table_to_latex_ci(methods, dgps, paper_means, paper_stds, fmt='{:.3f}')
    save_text(os.path.join(tables_dir, 'ci_length_paper.txt'), latex_paper)

    print('Saved tables to:', tables_dir)










=== n=1000 (1350 rows) ===
Saved tables to: outputs\exp1_backdoor_cate_attempt_1\tables
